<a href="https://colab.research.google.com/github/mahadikprasad15/ARENA/blob/main/Probes_for_finding_capitalization_feature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install transformer_lens

In [ ]:
import torch
import torch.nn as nn
from transformer_lens import HookedTransformer
from typing import List, Dict, Tuple
import numpy as np
import random
from dataclasses import dataclass
from tqdm import tqdm
import plotly.express as px

device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(42)
torch.manual_seed(42)

In [ ]:
model = HookedTransformer.from_pretrained('gpt2-small')
model.eval()

In [ ]:
@dataclass
class CapitalizationSample:

    prompt: str
    label: int  # 0 = not capitalized, 1 = capitalized
    target_word: str
    target_position: int  # token position to probe

def create_capitalization_dataset(
    n_samples: int = 1000,
    seed: int = 42
) -> Tuple[List[str], torch.Tensor, List[int]]:
    """
    Create balanced dataset for capitalization detection.

    Template: "The word [TARGET] is interesting"

    Returns:
        prompts: List of prompt strings
        labels: Tensor [n_samples] with 0/1
        target_positions: List of token positions (where TARGET is)

    Design decisions to think about:
    - How to ensure 50/50 balance?
    - Should you use the same words capitalized and uncapitalized?
    - How many unique words do you need?
    """
    random.seed(seed)

    words = [
        "apple", "river", "mountain", "coffee", "music",
        "ocean", "forest", "garden", "sunset", "window",
        "bridge", "castle", "desert", "island", "valley"
    ]


    prompt_template = 'This word {} is interesting'
    prompts_1 = [prompt_template.format(random.choice(words).capitalize()) for _ in range(int(n_samples / 2))]
    labels_1 = torch.ones(len(prompts_1), dtype = torch.long)
    prompts_0 = [prompt_template.format(random.choice(words)) for _ in range(int(n_samples / 2))]
    labels_0 = torch.zeros(len(prompts_1), dtype = torch.long)

    labels = torch.cat([labels_1, labels_0])
    prompts_1.extend(prompts_0)
    prompts = prompts_1

    target_positions = torch.cat([labels_1 * 3,labels_1 * 3]).tolist()

    return prompts,labels ,target_positions

In [ ]:
prompts, labels, positions = create_capitalization_dataset(100)
tokens = model.to_tokens(prompts)

positions_tensor = torch.tensor(positions)

dummy_activations = torch.randn(len(prompts), tokens.shape[1], 10)

selected_activations = dummy_activations[torch.arange(len(prompts)), positions_tensor]

print("Shape of selected activations:", selected_activations.shape)

In [ ]:
def cache_residual_activations(
    model: HookedTransformer,
    tokens: torch.Tensor,
    target_positions: List[int],
    layers: List[int] = None
) -> Dict[int, torch.Tensor]:

    layer_activations = {}
    if layers is None:
        layers = list(range(model.cfg.n_layers))

    target_positions = torch.tensor(target_positions)

    _, cache = model.run_with_cache(tokens)
    for layer in tqdm(layers):
      activation = cache[f'blocks.{layer}.hook_resid_post']

      activation = activation[torch.arange(tokens.size(0)), target_positions]
      layer_activations[f'blocks.{layer}.hook_resid_post'] = activation

    return layer_activations


In [ ]:
cache_dict = cache_residual_activations(model, tokens, positions)
train_acts_by_layers = {int(x.split('.')[1]): y for x,y in cache_dict.items()}

In [ ]:
class LinearProbe(nn.Module):
  def __init__(self, d_model, n_classes):
    super().__init__()

    self.linear = nn.Linear(d_model, n_classes)


  def forward(self, x):
    return self.linear(x)

In [ ]:
def split_data(
    activations_by_layer: Dict[int, torch.Tensor],
    labels: torch.Tensor,
    prompts: List[str],
    train_frac: float = 0.8,
    seed: int = 42
) -> Tuple[Dict, Dict, torch.Tensor, torch.Tensor, List[str], List[str]]:
    """
    Split activations, labels, and prompts into train/test sets.

    Returns:
        train_acts_by_layer: Dict[layer -> train_activations]
        test_acts_by_layer: Dict[layer -> test_activations]
        train_labels: Train labels
        test_labels: Test labels
        train_prompts: Train prompts (for later analysis)
        test_prompts: Test prompts

    CRITICAL: The split must be CONSISTENT across all layers!
    If sample 5 is in train set, it must be in train for ALL layers.

    STRATEGY:
    1. Generate random split indices once
    2. Apply same indices to all layers
    """


    # THINKING NUDGE 1: Random split
    # Create random permutation of indices
    # Split into train_indices and test_indices
    # Use torch.randperm()

    # THINKING NUDGE 2: Applying split
    # For each layer's activations, index with train_indices and test_indices
    # Same for labels and prompts

    # YOUR CODE HERE

    torch.manual_seed(seed)
    n_samples = labels.shape[0]
    split_point = int(train_frac * n_samples)


    shuffled_indices = torch.randperm(n_samples)
    train_indices = shuffled_indices[:split_point].tolist()

    test_indices = shuffled_indices[split_point:].tolist()


    train_acts_by_layer = {x: y[train_indices, :] for x,y in activations_by_layer.items()}
    test_acts_by_layer = {x: y[test_indices, :] for x,y in activations_by_layer.items()}
    train_labels = labels[train_indices]

    test_labels = labels[test_indices]

    train_prompts = [prompts[x] for x in train_indices]

    test_prompts = [prompts[x] for x in test_indices]

    return (train_acts_by_layer, test_acts_by_layer,
            train_labels, test_labels,
            train_prompts, test_prompts)

# Test
train_acts, test_acts, train_labels, test_labels, train_prompts, test_prompts = split_data(
    activations, labels[:20], prompts[:20], train_frac=0.8
)

assert train_labels.shape[0] == 16, f"Expected 16 train samples"
assert test_labels.shape[0] == 4, f"Expected 4 test samples"
assert len(train_prompts) == 16
assert all(train_acts[layer].shape[0] == 16 for layer in train_acts), "Inconsistent split!"

print(f"✓ Train size: {len(train_labels)}")
print(f"✓ Test size: {len(test_labels)}")
print(f"✓ Train label distribution: {train_labels.sum().item()}/{len(train_labels)}")

In [ ]:
def train_one_epoch(
    probe: nn.Module,
    train_activations: torch.Tensor,
    train_labels: torch.Tensor,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    batch_size: int = 64,
    device: str = "cpu"
) -> Tuple[float, float]:
    """
    Train probe for one epoch on cached activations.

    Returns:
        avg_loss: Average loss over epoch
        accuracy: Training accuracy

    THINKING:
    - Activations are already cached (not recomputing forward pass!)
    - Just training the probe weights
    - Need to create batches from cached activations
    """

    probe.train()
    train_activations = train_activations.to(device)
    train_labels = train_labels.to(device)

    n_samples = train_activations.shape[0]


    random_indices = torch.randperm(n_samples)
    batches = torch.split(random_indices,  batch_size)

    total_loss = 0
    total_correct = 0

    for batch in batches:
      batch_acts = train_activations[batch]
      batch_labels = train_labels[batch]

      optimizer.zero_grad()
      logits = probe(batch_acts)
      loss = criterion(logits, batch_labels)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()

      prediction = logits.argmax(dim = -1)
      total_correct += (prediction == batch_labels).sum().item()

    avg_loss = total_loss/ n_samples
    accuracy = total_correct / n_samples

    return avg_loss, accuracy

# Test (mini test with small data)
test_probe = LinearProbe(768, 2).to(device)
test_optimizer = torch.optim.AdamW(test_probe.parameters(), lr=1e-3)
test_criterion = nn.CrossEntropyLoss()

loss, acc = train_one_epoch(
    test_probe,
    train_acts[0],  # Layer 0 activations
    train_labels,
    test_optimizer,
    test_criterion,
    batch_size=8,
    device=device
)

assert isinstance(loss, float), "Loss should be a float"
assert isinstance(acc, float), "Accuracy should be a float"
assert 0 <= acc <= 1, f"Accuracy should be in [0,1], got {acc}"

print(f"✓ Single epoch complete")
print(f"✓ Loss: {loss:.4f}")
print(f"✓ Accuracy: {acc:.4f}")

In [ ]:
@torch.no_grad()
def evaluate_probe(
    probe: nn.Module,
    test_activations: torch.Tensor,
    test_labels: torch.Tensor,
    criterion: nn.Module,
    device: str = "cpu"
) -> Tuple[float, float]:
    """
    Evaluate probe on test set.

    Returns:
        test_loss: Average loss
        test_accuracy: Accuracy

    THINKING:
    - No gradients needed (@torch.no_grad())
    - Probe in eval mode (probe.eval())
    - Can process all at once (small test set) or in batches
    """
    probe.eval()
    test_activations = test_activations.to(device)
    test_labels = test_labels.to(device)

    outputs = probe(test_activations)
    test_loss = criterion(outputs, test_labels)
    predictions = outputs.argmax(dim = -1)
    correct = (predictions == test_labels).sum().item()
    test_accuracy = correct / len(test_labels)

    return test_loss, test_accuracy

# Test
test_loss, test_acc = evaluate_probe(
    test_probe,
    test_acts[0],
    test_labels,
    test_criterion,
    device=device
)

print(f"✓ Evaluation complete")
print(f"✓ Test loss: {test_loss:.4f}")
print(f"✓ Test accuracy: {test_acc:.4f}")

In [ ]:
def train_probe_full(
    probe: nn.Module,
    train_activations: torch.Tensor,
    train_labels: torch.Tensor,
    test_activations: torch.Tensor,
    test_labels: torch.Tensor,
    n_epochs: int = 100,
    lr: float = 1e-3,
    batch_size: int = 64,
    device: str = "cpu",
    verbose: bool = True
) -> Dict[str, List[float]]:
    """
    Full training loop with history tracking.

    Returns:
        history: Dict with keys 'train_loss', 'train_acc', 'test_loss', 'test_acc'
                 Each is a list of length n_epochs

    STRATEGY:
    - Setup optimizer and criterion
    - For each epoch:
        - Train one epoch
        - Evaluate on test
        - Store metrics
        - Optionally print progress
    """

    # THINKING NUDGE 1: Setup
    # Create optimizer (AdamW is good default)
    # Create criterion (CrossEntropyLoss for classification)
    # Move probe to device

    # THINKING NUDGE 2: History tracking
    # Initialize dict with empty lists
    # Append metrics after each epoch

    # THINKING NUDGE 3: Early stopping?
    # For this simple task, probably not needed
    # But could add: if test_acc hasn't improved in 10 epochs, stop

    # YOUR CODE HERE

    optimizer = torch.optim.Adam(probe.parameters(), lr = lr)
    criterion = nn.CrossEntropyLoss()
    probe.to(device)

    history = {'train_loss':[], 'train_acc': [], 'test_loss':[], 'test_acc':[]}

    for epoch in tqdm(range(n_epochs)):

      probe.train()

      random_indices = torch.randperm(train_activations.size(0))
      total_loss = 0
      total_correct = 0
      batches = torch.split(random_indices, batch_size)

      for batch in batches:
        optimizer.zero_grad()

        activation_data = train_activations[batch]
        label_data = train_labels[batch]
        output = probe(activation_data)
        loss = criterion(output, label_data)
        predictions = output.argmax(dim= -1)

        total_loss += loss.item() * len(batch)
        total_correct += (predictions == label_data).sum().item()

        loss.backward()
        optimizer.step()

      avg_loss = total_loss / train_activations.size(0)
      history['train_loss'].append(avg_loss)

      avg_accuracy = total_correct/ train_activations.size(0)
      history['train_acc'].append(avg_accuracy)

      probe.eval()
      with torch.no_grad():
        test_output = probe(test_activations)
        test_loss = criterion(test_output, test_labels)
        test_loss = test_loss/ test_activations.size(0)

        predictions = test_output.argmax(dim = -1)
        correct_predictions = (predictions == test_labels).sum().item()
        test_accuracy = correct_predictions / test_activations.size(0)

        history['test_loss'].append(test_loss.item())
        history['test_acc'].append(test_accuracy)


    return history

# Test with small dataset
print("Training probe on layer 0...")
history = train_probe_full(
    LinearProbe(768, 2).to(device),
    train_acts[0],
    train_labels,
    test_acts[0],
    test_labels,
    n_epochs=50,
    lr=1e-3,
    device=device,
    verbose=True
)

assert len(history['train_loss']) == 50
assert len(history['test_acc']) == 50

print(f"\n✓ Training complete!")
print(f"✓ Final train acc: {history['train_acc'][-1]:.3f}")
print(f"✓ Final test acc: {history['test_acc'][-1]:.3f}")

In [ ]:
import pandas as pd

# Create a DataFrame for plotting
df_history = pd.DataFrame({
    'Epoch': list(range(1, len(history['train_loss']) + 1)),
    'Train Loss': history['train_loss']
})

# Use px.line to plot the training loss over epochs
fig = px.line(df_history, x='Epoch', y='Train Loss', title='Training Loss Over Epochs')
fig.update_layout(yaxis_title='Loss')
fig.show()

df_history = pd.DataFrame({
    'Epoch': list(range(1, len(history['test_loss']) + 1)),
    'Train Loss': history['test_loss']
})

# Use px.line to plot the training loss over epochs
fig = px.line(df_history, x='Epoch', y='Train Loss', title='Training Loss Over Epochs')
fig.update_layout(yaxis_title='Loss')
fig.show()

In [ ]:
def train_probes_all_layers(
    train_acts_by_layer: Dict[int, torch.Tensor],
    test_acts_by_layer: Dict[int, torch.Tensor],
    train_labels: torch.Tensor,
    test_labels: torch.Tensor,
    d_model: int = 768,
    n_classes: int = 2,
    n_epochs: int = 50,
    device: str = "cpu"
) -> Dict[int, Tuple[nn.Module, Dict]]:
    """
    Train a separate probe for each layer.

    Returns:
        Dict mapping layer_idx -> (trained_probe, history)

    THINKING:
    - Each layer gets its own fresh probe
    - Training is independent per layer
    - Can parallelize, but sequential is fine for 12 layers
    """
    results = {}

    # THINKING NUDGE 1: Iteration
    # Loop over layers in train_acts_by_layer
    # For each layer, create new probe and train it

    # THINKING NUDGE 2: Progress tracking
    # Use tqdm for progress bar (optional but nice)
    # Print which layer you're on

    # YOUR CODE HERE

    for layer in tqdm(range(model.cfg.n_layers)):


    return results

# Test with full dataset
print("\n" + "="*50)
print("TRAINING PROBES FOR ALL LAYERS")
print("="*50)

# First, get activations for ALL layers on full dataset
all_layers = list(range(0, 12, 2))  # [0, 2, 4, 6, 8, 10]

# (You need to run the full pipeline here - create dataset, tokenize, cache activations)
# I'll leave this for you to connect the pieces



In [ ]:
prompts, labels, positions = create_capitalization_dataset(100)
tokens = model.to_tokens(prompts)

cache_dict = cache_residual_activations(model, tokens, positions)
acts_by_layers = {int(x.split('.')[1]): y for x,y in cache_dict.items()}


train_acts, test_acts, train_labels, test_labels, train_prompts, test_prompts = split_data(
    activations, labels[:20], prompts[:20], train_frac=0.8
)

In [ ]:
acts_by_layers